# Feature Engineering

In [36]:
import polars as pl
from pathlib import Path
import tomllib
import os

In [37]:
config_file_path = Path("src/config.toml")

with config_file_path.open("rb") as config_file:
    config = tomllib.load(config_file)

COMPANIES = config['companies']

In [38]:
company = next(company for company in COMPANIES)

In [39]:
lf = pl.scan_parquet(f'ETL/data/silver/parquet/{company}_share_prices_cleaned.parquet')

In [40]:
lf = lf.with_columns(
    # Log Return
    (pl.col('Adj. Close') / pl.col('Adj. Close').shift(1)).log().alias(f'Log Return')
)

for window in [5, 10, 20]:
    lf = lf.with_columns(
        # Rolling Log Returns
        (pl.col('Adj. Close') / pl.col('Adj. Close').shift(window)).log().alias(f'Log Return {window}d'),

        # Volatility
        pl.col('Log Return').rolling_std(window).log1p().alias(f'Volatility {window}d'),

        # Moving Averages
        pl.col("Adj. Close").rolling_mean(5).alias(f"Moving Average {window}d"),

        # Momentum Pct. Change
        ((pl.col("Adj. Close") / pl.col("Adj. Close").shift(window)) - 1).alias(f"Momentum Pct. {window}d"),

        # Log Volume Ratio
        ((pl.col("Volume") / pl.col("Volume").rolling_mean(window))).log().alias(f"Log Volume Ratio {window}d"),
    )

    lf = lf.with_columns(
        # Log MA Ratio
        (pl.col("Adj. Close") / pl.col(f"Moving Average {window}d")).log().alias(f"Log MA Ratio {window}d")
    )

lf = lf.with_columns(
    # Intraday Returns
    ((pl.col('Close') / pl.col('Open')) - 1).alias('Intraday Pct. Return'),

    # Ranges
    (pl.col('High') - pl.col('Low')).alias('Range'),
    ((pl.col('High') - pl.col('Low')) / pl.col('Close')).alias('Range Pct.'),

    # Close Position
    (((pl.col('Close') - pl.col('Low')) / (pl.col('High') - pl.col('Low'))) - 0.5).alias('Close Position'),

    # Log Volume Change
    (pl.col("Volume") / pl.col("Volume").shift(1)).log().tanh().alias("Log Volume Change"),

    # Log Market Cap
    (pl.col("Adj. Close") * pl.col("Shares Outstanding")).log().alias("Log Market Cap"),

    # Dilution / Issuance
    (pl.col("Shares Outstanding") / pl.col("Shares Outstanding").shift(1) - 1).alias('Delta Pct. Dilution / Issuance'),

    # Volume Return Interaction
    (pl.col("Log Return") * pl.col("Log Volume Ratio 5d")).tanh().alias("Interaction Return Volume 5d"),
    (pl.col("Log Return") * pl.col("Log Volume Ratio 10d")).tanh().alias("Interaction Return Volume 10d"),
    (pl.col("Log Return") * pl.col("Log Volume Ratio 20d")).tanh().alias("Interaction Return Volume 20d"),

    # Volume Volatility Interaction
    (pl.col("Volatility 5d") * pl.col("Log Volume Ratio 5d")).tanh().alias("Interaction Volatility Volume 5d"),
    (pl.col("Volatility 10d") * pl.col("Log Volume Ratio 10d")).tanh().alias("Interaction Volatility Volume 10d"),
    (pl.col("Volatility 20d") * pl.col("Log Volume Ratio 20d")).tanh().alias("Interaction Volatility Volume 20d"),

    # Momentum Volatility Interaction
    (pl.col("Momentum Pct. 5d") * pl.col("Volatility 5d")).tanh().alias("Interaction Momentum Volatility 5d"),
    (pl.col("Momentum Pct. 10d") * pl.col("Volatility 10d")).tanh().alias("Interaction Momentum Volume 10d"),
    (pl.col("Momentum Pct. 20d") * pl.col("Volatility 20d")).tanh().alias("Interaction Momentum Volume 20d"),

    # Target Engineering
    (pl.col("Adj. Close").shift(-1) / pl.col("Adj. Close") - 1).alias("Target Return Metric"),
    ((pl.col("Adj. Close").shift(-1) / pl.col("Adj. Close") - 1) > 0).alias("Target Return Class")
)

In [41]:
with pl.Config(tbl_rows=-1):
    display(lf.head(25).collect())

Open,High,Low,Close,Adj. Close,Volume,Shares Outstanding,Log Return,Log Return 5d,Volatility 5d,Moving Average 5d,Momentum Pct. 5d,Log Volume Ratio 5d,Log MA Ratio 5d,Log Return 10d,Volatility 10d,Moving Average 10d,Momentum Pct. 10d,Log Volume Ratio 10d,Log MA Ratio 10d,Log Return 20d,Volatility 20d,Moving Average 20d,Momentum Pct. 20d,Log Volume Ratio 20d,Log MA Ratio 20d,Intraday Pct. Return,Range,Range Pct.,Close Position,Log Volume Change,Log Market Cap,Delta Pct. Dilution / Issuance,Interaction Return Volume 5d,Interaction Return Volume 10d,Interaction Return Volume 20d,Interaction Volatility Volume 5d,Interaction Volatility Volume 10d,Interaction Volatility Volume 20d,Interaction Momentum Volatility 5d,Interaction Momentum Volume 10d,Interaction Momentum Volume 20d,Target Return Metric,Target Return Class
f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool
119.5,122.25,119.3,119.68,119.68,115413880,9.9800e9,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.001506,2.95,0.024649,-0.371186,null,27.80867,null,null,null,null,null,null,null,null,null,null,-0.027323,false
120.83,121.42,113.98,116.41,116.41,149533580,9.9800e9,-0.027703,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,-0.03658,7.44,0.063912,-0.173387,0.253357,27.780967,0.0,null,null,null,null,null,null,null,null,null,0.015119,true
118.45,119.7,117.55,118.17,118.17,84365900,9.9755e9,0.015006,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,-0.002364,2.15,0.018194,-0.211628,-0.517089,27.795524,-0.000449,null,null,null,null,null,null,null,null,null,0.015232,true
120.0,121.21,119.1,119.97,119.97,101331040,9.9755e9,0.015117,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,-0.00025,2.11,0.017588,-0.087678,0.181206,27.810642,0.0,null,null,null,null,null,null,null,null,null,0.004501,true
120.85,121.02,119.1,120.51,120.51,76635940,9.9755e9,0.004491,null,null,118.948,null,-0.319228,0.013046,null,null,118.948,null,null,0.013046,null,null,118.948,null,null,0.013046,-0.002813,1.92,0.015932,0.234375,-0.272282,27.815133,0.0,-0.001434,null,null,null,null,null,null,null,null,-0.01419,false
122.16,122.24,118.15,118.8,118.8,112912900,9.9755e9,-0.014291,-0.00738,0.018762,118.772,-0.007353,0.073077,0.000236,null,null,118.772,null,null,0.000236,null,null,118.772,null,null,0.000236,-0.027505,4.09,0.034428,-0.341076,0.369247,27.800841,0.0,-0.001044,null,null,0.001371,null,null,-0.000138,null,null,-0.026094,false
118.61,118.67,115.3,115.7,115.7,105388920,9.9755e9,-0.026441,-0.006118,0.018336,118.63,-0.006099,0.091988,-0.025009,null,null,118.63,null,null,-0.025009,null,null,118.63,null,null,-0.025009,-0.024534,3.37,0.029127,-0.381306,-0.06885,27.774401,0.0,-0.002432,null,null,0.001687,null,null,-0.000112,null,null,0.025411,true
116.5,119.59,115.5,118.64,118.64,91831860,9.9755e9,0.025093,0.003969,0.020867,118.724,0.003977,-0.061124,-0.000708,null,null,118.724,null,null,-0.000708,null,null,118.724,null,null,-0.000708,0.018369,4.09,0.034474,0.267726,-0.136834,27.799494,0.0,-0.001534,null,null,-0.001276,null,null,0.000083,null,null,0.04265,true
120.99,123.75,119.8,123.7,123.7,190692220,9.9755e9,0.041766,0.030618,0.027502,119.47,0.031091,0.501456,0.034794,null,null,119.47,null,null,0.034794,null,null,119.47,null,null,0.034794,0.022399,3.95,0.031932,0.487342,0.623494,27.841259,0.0,0.020941,null,null,0.01379,null,null,0.000855,null,null,-0.07599,false


In [42]:
lf = lf.drop([
    "Open", 
    "High",
    "Low",
    "Close",
    "Adj. Close",
    "Moving Average 5d",
    "Moving Average 10d",
    "Moving Average 20d",
    "Volume",
    "Range",
    "Shares Outstanding"
])

lf = lf.drop_nulls()

In [43]:
with pl.Config(tbl_rows=-1):
    display(lf.head(21).collect())

Log Return,Log Return 5d,Volatility 5d,Momentum Pct. 5d,Log Volume Ratio 5d,Log MA Ratio 5d,Log Return 10d,Volatility 10d,Momentum Pct. 10d,Log Volume Ratio 10d,Log MA Ratio 10d,Log Return 20d,Volatility 20d,Momentum Pct. 20d,Log Volume Ratio 20d,Log MA Ratio 20d,Intraday Pct. Return,Range Pct.,Close Position,Log Volume Change,Log Market Cap,Delta Pct. Dilution / Issuance,Interaction Return Volume 5d,Interaction Return Volume 10d,Interaction Return Volume 20d,Interaction Volatility Volume 5d,Interaction Volatility Volume 10d,Interaction Volatility Volume 20d,Interaction Momentum Volatility 5d,Interaction Momentum Volume 10d,Interaction Momentum Volume 20d,Target Return Metric,Target Return Class
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool
0.006782,0.007115,0.013019,0.00714,0.082324,0.01505,0.046485,0.01001,0.047582,0.183791,0.01505,0.013528,0.024874,0.01362,-0.100949,0.01505,0.009067,0.020196,0.361224,0.030597,27.821749,0.0,0.000558,0.001247,-0.000685,0.001072,0.00184,-0.002511,0.000093,0.000476,0.000339,0.009562,true
0.009517,0.038453,0.001973,0.039202,0.011613,0.016864,0.055225,0.010016,0.056778,0.143946,0.016864,0.050748,0.024048,0.052057,-0.078341,0.016864,0.008067,0.022863,-0.135714,-0.010607,27.831266,0.0,0.000111,0.00137,-0.000746,0.000023,0.001442,-0.001884,0.000077,0.000569,0.001252,0.019842,true
0.019647,0.053445,0.005096,0.054899,-0.028483,0.025774,0.060565,0.010648,0.062436,0.043148,0.025774,0.055389,0.024192,0.056952,-0.153523,0.025774,0.008152,0.013131,0.439024,-0.077386,27.850913,0.0,-0.00056,0.000848,-0.003016,-0.000145,0.000459,-0.003714,0.00028,0.000665,0.001378,-0.020496,false
-0.020709,0.02399,0.014995,0.02428,0.148886,0.000311,0.032904,0.013532,0.033452,0.245492,0.000311,0.019562,0.02454,0.019755,0.092194,0.000311,-0.02128,0.03384,-0.449275,0.241372,27.830204,0.0,-0.003083,-0.005084,-0.001909,0.002233,0.003322,0.002262,0.000364,0.000453,0.000485,-0.004087,false
-0.004095,0.011142,0.015241,0.011204,-0.365792,-0.00599,0.023753,0.013704,0.024038,-0.324637,-0.00599,0.010976,0.02455,0.011036,-0.476176,-0.00599,-0.007413,0.016251,-0.333333,-0.521769,27.826109,0.0,0.001498,0.001329,0.00195,-0.005575,-0.004449,-0.01169,0.000171,0.000329,0.000271,-0.006156,false
-0.006175,-0.001815,0.015371,-0.001814,-0.107638,-0.011806,0.005299,0.013468,0.005313,-0.11364,-0.011806,0.019093,0.024365,0.019276,-0.234661,-0.011806,-0.014727,0.01982,-0.3375,0.21533,27.819934,0.0,0.000665,0.000702,0.001449,-0.001654,-0.00153,-0.005717,-0.000028,0.000072,0.00047,-0.004707,false
-0.004718,-0.01605,0.014392,-0.015922,0.20468,-0.013336,0.022403,0.011256,0.022656,0.186684,-0.013336,0.040815,0.02358,0.041659,0.116412,-0.013336,0.002245,0.034683,0.461722,0.335235,27.815216,0.0,-0.000966,-0.000881,-0.000549,0.002946,0.002101,0.002745,-0.000229,0.000255,0.000982,-0.003817,false
-0.003824,-0.039522,0.00719,-0.038751,-0.215226,-0.009203,0.013923,0.011371,0.01402,-0.234449,-0.009203,0.011898,0.022992,0.011969,-0.327691,-0.009203,0.007046,0.024488,-0.108844,-0.429888,27.811392,0.0,0.000823,0.000897,0.001253,-0.001547,-0.002666,-0.007534,-0.000279,0.000159,0.000275,0.017158,true
0.017013,-0.0018,0.009707,-0.001798,-0.03168,0.008173,0.02219,0.012218,0.022438,-0.131397,0.008173,-0.012855,0.021319,-0.012773,-0.157422,0.008173,0.010927,0.018097,0.5,0.099771,27.828404,0.0,-0.000539,-0.002235,-0.002678,-0.000308,-0.001605,-0.003356,-0.000017,0.000274,-0.000272,0.01171,true


In [44]:
SILVER_CSV_DIR = os.path.join('ETL', 'data', 'silver', 'csv')
os.makedirs(SILVER_CSV_DIR, exist_ok=True)
filepath = os.path.join(SILVER_CSV_DIR, 'silver_test.csv')
lf.head(50).collect().write_csv(filepath)